# Module 6: AI Gateways and MLOps for LLM Deployment

Build a centralized gateway for model routing, cost control, and provider failover.


## 1. Setup

Load the gateway core and sample inputs.


In [1]:
import sys
from pathlib import Path

module_dir = Path.cwd()
if str(module_dir) not in sys.path:
    sys.path.append(str(module_dir))

from ai_gateway_core import AIGateway, sample_receipts

print("✓ Setup complete")


✓ Setup complete


## 2. Initialize Gateway

This simulates a centralized AI gateway (LiteLLM/Kong style).


In [2]:
gateway = AIGateway()
receipts = sample_receipts()

print("✓ Gateway initialized")
print("Routes:", gateway.primary_route)


2026-03-01 10:45:34 - ai_gateway_core - INFO - AIGateway initialized successfully


✓ Gateway initialized
Routes: {'simple': ['groq', 'openai', 'azure'], 'moderate': ['openai', 'azure', 'anthropic'], 'complex': ['openai', 'azure', 'anthropic']}


### Inspect Sample Receipts


In [3]:
receipts


{'SIMPLE-COFFEE': {'id': 'SIMPLE-COFFEE',
  'category': 'Coffee',
  'total': 6.5,
  'note': 'Team coffee during sprint planning',
  'attendees': 1},
 'MODERATE-TRAVEL': {'id': 'MODERATE-TRAVEL',
  'category': 'Travel',
  'total': 85.0,
  'note': 'Airport rideshare to client office',
  'attendees': 1},
 'COMPLEX-DINNER': {'id': 'COMPLEX-DINNER',
  'category': 'Client Dinner',
  'total': 420.0,
  'note': 'Client dinner with exception check for VP attendees',
  'attendees': 5}}

## 3. Smart Routing

- simple -> low-cost model
- moderate/complex -> stronger models


In [4]:
for rid, receipt in receipts.items():
    result = gateway.route_and_infer(receipt)
    print(rid, "->", result.provider, result.model, "cost", result.estimated_cost_usd)


2026-03-01 10:45:34 - ai_gateway_core - INFO - Routing and inferring for receipt: SIMPLE-COFFEE
2026-03-01 10:45:34 - ai_gateway_core - INFO - Classified receipt complexity as: simple
2026-03-01 10:45:34 - ai_gateway_core - INFO - Successfully routed to groq using llama-3.1-8b-instant for simple request
2026-03-01 10:45:34 - ai_gateway_core - INFO - Routing and inferring for receipt: MODERATE-TRAVEL
2026-03-01 10:45:34 - ai_gateway_core - INFO - Classified receipt complexity as: simple
2026-03-01 10:45:34 - ai_gateway_core - INFO - Successfully routed to groq using llama-3.1-8b-instant for simple request
2026-03-01 10:45:34 - ai_gateway_core - INFO - Routing and inferring for receipt: COMPLEX-DINNER
2026-03-01 10:45:34 - ai_gateway_core - INFO - Classified receipt complexity as: complex
2026-03-01 10:45:34 - ai_gateway_core - INFO - Successfully routed to openai using gpt-4o-mini for complex request


SIMPLE-COFFEE -> groq llama-3.1-8b-instant cost 0.0243
MODERATE-TRAVEL -> groq llama-3.1-8b-instant cost 0.0243
COMPLEX-DINNER -> openai gpt-4o-mini cost 0.2406


## 4. Fallback Simulation

Force provider outages and verify automatic rerouting.


In [5]:
# simulate OpenAI outage
gateway.set_provider_health("openai", False)

complex_receipt = receipts["COMPLEX-DINNER"]
result_failover_1 = gateway.route_and_infer(complex_receipt)
print("OpenAI down ->", result_failover_1.to_dict())

# simulate Azure outage too
gateway.set_provider_health("azure", False)
result_failover_2 = gateway.route_and_infer(complex_receipt)
print("OpenAI + Azure down ->", result_failover_2.to_dict())

# bring providers back
gateway.set_provider_health("openai", True)
gateway.set_provider_health("azure", True)


2026-03-01 10:45:34 - ai_gateway_core - INFO - Setting provider openai health to False
2026-03-01 10:45:34 - ai_gateway_core - INFO - Provider openai health updated successfully
2026-03-01 10:45:34 - ai_gateway_core - INFO - Routing and inferring for receipt: COMPLEX-DINNER
2026-03-01 10:45:34 - ai_gateway_core - INFO - Classified receipt complexity as: complex
2026-03-01 10:45:34 - ai_gateway_core - INFO - Successfully routed to azure using gpt-4o-mini for complex request
2026-03-01 10:45:34 - ai_gateway_core - INFO - Setting provider azure health to False
2026-03-01 10:45:34 - ai_gateway_core - INFO - Provider azure health updated successfully
2026-03-01 10:45:34 - ai_gateway_core - INFO - Routing and inferring for receipt: COMPLEX-DINNER
2026-03-01 10:45:34 - ai_gateway_core - INFO - Classified receipt complexity as: complex
2026-03-01 10:45:34 - ai_gateway_core - INFO - Successfully routed to anthropic using claude-3-5-haiku for complex request
2026-03-01 10:45:34 - ai_gateway_core

OpenAI down -> {'ok': True, 'provider': 'azure', 'model': 'gpt-4o-mini', 'reason': 'Routed via complex policy', 'tokens': 401, 'estimated_cost_usd': '0.2606'}
OpenAI + Azure down -> {'ok': True, 'provider': 'anthropic', 'model': 'claude-3-5-haiku', 'reason': 'Routed via complex policy', 'tokens': 401, 'estimated_cost_usd': '0.3208'}


## 5. Logging and Cost Visibility

Centralized logs and cumulative cost are key gateway benefits.


In [6]:
print("Total logged events:", len(gateway.logs))
print("Total estimated cost USD:", gateway.total_cost())

for row in gateway.logs[-6:]:
    print(row)


2026-03-01 10:45:34 - ai_gateway_core - INFO - Calculating total cost across all gateway operations
2026-03-01 10:45:34 - ai_gateway_core - INFO - Total cost calculated: $0.8706


Total logged events: 8
Total estimated cost USD: 0.8706
{'time': '2026-03-01T15:45:34Z', 'provider': 'openai', 'model': 'gpt-4o-mini', 'complexity': 'complex', 'ok': True, 'event': 'success', 'tokens': 401, 'estimated_cost_usd': '0.2406'}
{'time': '2026-03-01T15:45:34Z', 'provider': 'openai', 'model': 'gpt-4o-mini', 'complexity': 'complex', 'ok': False, 'event': 'provider_unhealthy', 'tokens': 401, 'estimated_cost_usd': '0.0000'}
{'time': '2026-03-01T15:45:34Z', 'provider': 'azure', 'model': 'gpt-4o-mini', 'complexity': 'complex', 'ok': True, 'event': 'success', 'tokens': 401, 'estimated_cost_usd': '0.2606'}
{'time': '2026-03-01T15:45:34Z', 'provider': 'openai', 'model': 'gpt-4o-mini', 'complexity': 'complex', 'ok': False, 'event': 'provider_unhealthy', 'tokens': 401, 'estimated_cost_usd': '0.0000'}
{'time': '2026-03-01T15:45:34Z', 'provider': 'azure', 'model': 'gpt-4o-mini', 'complexity': 'complex', 'ok': False, 'event': 'provider_unhealthy', 'tokens': 401, 'estimated_cost_usd': '0.00

## 6. Build Artifact: gateway_config.yaml

Generate the production-style gateway routing/failover config.


In [7]:
cfg = gateway.generate_gateway_config_yaml()
print(cfg)

Path("gateway_config.yaml").write_text(cfg)
print("✓ Wrote gateway_config.yaml")


2026-03-01 10:45:34 - ai_gateway_core - INFO - Generating gateway configuration YAML
2026-03-01 10:45:34 - ai_gateway_core - INFO - Gateway configuration YAML generated successfully


version: 1
gateway:
  logging: enabled
  cost_tracking: enabled
  timeout_seconds: 30
providers:
  - name: openai
    model: gpt-4o-mini
    role: primary
  - name: azure
    model: gpt-4o-mini
    role: failover
  - name: anthropic
    model: claude-3-5-haiku
    role: failover
  - name: groq
    model: llama-3.1-8b-instant
    role: low_cost
routing:
  simple: [groq, openai, azure]
  moderate: [openai, azure, anthropic]
  complex: [openai, azure, anthropic]
fallback:
  strategy: ordered_failover
  on_error: true

✓ Wrote gateway_config.yaml


## 7. Assertions (Smoke Checks)


In [8]:
# Simple receipt should route to cheap provider by default.
g2 = AIGateway()
simple = g2.route_and_infer(receipts["SIMPLE-COFFEE"])
assert simple.provider == "groq"

# Complex fallback chain should avoid unhealthy primary.
g3 = AIGateway()
g3.set_provider_health("openai", False)
fallback = g3.route_and_infer(receipts["COMPLEX-DINNER"])
assert fallback.provider in {"azure", "anthropic"}

print("✓ Module 6 gateway smoke checks passed")


2026-03-01 10:45:34 - ai_gateway_core - INFO - AIGateway initialized successfully
2026-03-01 10:45:34 - ai_gateway_core - INFO - Routing and inferring for receipt: SIMPLE-COFFEE
2026-03-01 10:45:34 - ai_gateway_core - INFO - Classified receipt complexity as: simple
2026-03-01 10:45:34 - ai_gateway_core - INFO - Successfully routed to groq using llama-3.1-8b-instant for simple request
2026-03-01 10:45:34 - ai_gateway_core - INFO - AIGateway initialized successfully
2026-03-01 10:45:34 - ai_gateway_core - INFO - Setting provider openai health to False
2026-03-01 10:45:34 - ai_gateway_core - INFO - Provider openai health updated successfully
2026-03-01 10:45:34 - ai_gateway_core - INFO - Routing and inferring for receipt: COMPLEX-DINNER
2026-03-01 10:45:34 - ai_gateway_core - INFO - Classified receipt complexity as: complex
2026-03-01 10:45:34 - ai_gateway_core - INFO - Successfully routed to azure using gpt-4o-mini for complex request


✓ Module 6 gateway smoke checks passed
